In [1]:
!pip install ultralytics

from IPython import display
display.clear_output()

import yaml
from ultralytics import YOLO

import random
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import pathlib
%matplotlib inline
from collections import Counter
import sys
import pandas as pd
import numpy as np
import random
import os
import shutil
import torch
import glob
from tqdm import tqdm
import os
import cv2
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import traceback
import importlib

In [2]:
!rm /home/smile/work/pbs06/test/labels.cache
!rm /home/smile/work/pbs06/train/labels.cache
!rm /home/smile/work/pbs06/val/labels.cache

rm: cannot remove '/home/smile/work/pbs06/test/labels.cache': No such file or directory


In [3]:
WORKING_DIR = "/home/smile/work/pbs06/"
TRAIN_DIR = os.path.join( WORKING_DIR, 'train')
VAL_DIR = os.path.join( WORKING_DIR, 'val')
TEST_DIR = os.path.join( WORKING_DIR, 'test')


TRAIN_IMAGE_DIR = os.path.join( TRAIN_DIR, 'images')
VAL_IMAGE_DIR = os.path.join( VAL_DIR, 'images')
TEST_IMAGE_DIR = os.path.join( TEST_DIR, 'images')


TRAIN_LABEL_DIR = os.path.join( TRAIN_DIR, 'labels')
VAL_LABEL_DIR = os.path.join( VAL_DIR, 'labels')
TEST_LABEL_DIR = os.path.join( TEST_DIR, 'labels')


# SOURCE_IMAGE_DIR = '/home/smile/work/pbs/dataset/u2labeler/48' # U2Labeler 용
# SOURCE_IMAGE_DIR = '/home/smile/work/pbs/dataset/u2labeler/48_crop' # U2Labeler-crop 용
# SOURCE_IMAGE_DIR = '/home/smile/work/pbs/dataset/di60' # DI60 검사모드용
# SOURCE_IMAGE_DIR = '/home/smile/work/pbs/dataset/di60_scan' # DI60 스캔모드용

YOLO_DIR = os.path.join( WORKING_DIR, 'yolov5')

TRAIN_RATE = 0.7 # 트레이닝 데이터 비중
IMAGEFILE_EXT = '.jpg'

In [4]:
# Load the YAML file
with open(os.path.join(YOLO_DIR, "data/blood_cell.yaml"), 'r') as f:
    data_yaml = yaml.safe_load(f)

# update YAML with absolute path to kaggle data. You must use absolute path, relative path won't work
# data_yaml['train'] = '/kaggle/input/blood-cell-detection-datatset/train/images'
# data_yaml['val'] = '/kaggle/input/blood-cell-detection-datatset/valid/images'

# # write to disk
# with open('data.yaml', 'w') as f:
#     yaml.dump(data_yaml, f)

In [5]:
data_yaml

{'train': '/home/smile/work/pbs06/train/images',
 'val': '/home/smile/work/pbs06/val/images',
 'test': '/home/smile/work/pbs06/test/images',
 'nc': 25,
 'names': ['BandNeutrophil',
  'Basophil',
  'Blast',
  'Echinocyte',
  'Elliptocyte',
  'Eosinophil',
  'GiantPlt',
  'Lymphocyte',
  'Monocyte',
  'Myelocyte',
  'Nucleated',
  'PLT',
  'RBC',
  'Reticulocyte',
  'Schistocyte',
  'SegNeutrophil',
  'Smudge',
  'Stomatocyte',
  'TargetCell',
  'TearDropCell',
  'ToxicGranule',
  'ToxicVacuole',
  'clump-PLT',
  'degeneWBC',
  'hyperSeg.']}

In [6]:
data_dir = Path(WORKING_DIR)
images_dir = data_dir / "train" / "images"
labels_dir = data_dir / "train" / "labels"

class_names = data_yaml['names']

# Read the image file paths and annotations
image_paths = list(images_dir.glob("*.jpg"))
label_paths = sorted(labels_dir.glob("*.txt"))

In [7]:
resolutions = []

for image_path in tqdm(image_paths):
    img = cv2.imread(str(image_path))
    h, w, _ = img.shape
    resolutions.append((w, h))

unique_resolutions = set(resolutions)
print("Unique resolutions:", unique_resolutions)

100%|██████████| 18958/18958 [00:12<00:00, 1468.38it/s]

Unique resolutions: {(368, 374)}


In [8]:
data = []

for file in label_paths:
    with open(file) as f:
        lines = f.readlines()
        num_lines = len(lines)
        unique_values = len(set(list(map(lambda x: x.split()[0], lines))))
        
        data.append([file, num_lines, unique_values])

df = pd.DataFrame(data, columns=['file', 'num_lines', 'unique_values'])

In [9]:
import torch

# CUDA 메모리 사용량 출력 함수
def print_cuda_memory_usage():
    if torch.cuda.is_available():
        total_memory = torch.cuda.get_device_properties(0).total_memory
        reserved_memory = torch.cuda.memory_reserved(0)
        allocated_memory = torch.cuda.memory_allocated(0)
        free_memory = reserved_memory - allocated_memory
        
        print(f"Total Memory: {total_memory / (1024 ** 3):.2f} GB")
        print(f"Reserved Memory: {reserved_memory / (1024 ** 3):.2f} GB")
        print(f"Allocated Memory: {allocated_memory / (1024 ** 3):.2f} GB")
        print(f"Free Memory: {free_memory / (1024 ** 3):.2f} GB")
    else:
        print("CUDA is not available.")

# GPU 메모리 사용량 출력
print_cuda_memory_usage()

Total Memory: 19.71 GB
Reserved Memory: 0.00 GB
Allocated Memory: 0.00 GB
Free Memory: 0.00 GB


In [12]:
# utils 모듈 경로 추가
sys.path.append('/home/smile/work/pbs06/yolov5')

from utils.dataloaders import create_dataloader
from utils.torch_utils import EarlyStopping

# 사전 학습된 모델을 로드
baseline_model = YOLO('yolov10s.pt')

new_data_yaml = {
    'train': '/home/smile/work/pbs06/train/images',
    'val': '/home/smile/work/pbs06/val/images',
    'test': '/home/smile/work/pbs06/test/images',
    'nc': 25,
    'names': [ 'BandNeutrophil', 'Basophil', 'Blast', 'Echinocyte', 'Elliptocyte', 'Eosinophil', 'GiantPlt', 'Lymphocyte', 'Monocyte', 'Myelocyte', 'Nucleated', 'PLT', 'RBC', 'Reticulocyte', 'Schistocyte', 'SegNeutrophil', 'Smudge', 'Stomatocyte', 'TargetCell', 'TearDropCell', 'ToxicGranule', 'ToxicVacuole', 'clump-PLT', 'degeneWBC', 'hyperSeg.'] 
}

YOLO_DIR = '/home/smile/work/pbs06/yolov5'
# 기존 하이퍼파라미터 설정에서 튜닝된 값으로 변경
params = {
    'data': os.path.join(YOLO_DIR, "data/blood_cell.yaml"),
    'imgsz': 384,
    'epochs': 800,
    'optimizer': 'AdamW',
    'pretrained': True,
    'lr0': 0.0036365501465160546,  # 튜닝된 초기 학습률
    'lrf': 0.0006434696163064856,   # 튜닝된 최종 학습률
    'cos_lr': True,
    'label_smoothing': 0.1,
    'patience': 100,
    'freeze': [0],
    'save_period': -1,
    'seed': 0,
    'batch': 64,  # 튜닝된 배치 크기
    'momentum': 0.9122194860814927,  # 튜닝된 모멘텀
    'weight_decay': 0.00019315779508838763  # 튜닝된 가중치 감소율
}

# 학습률 스케줄러 설정
def one_cycle(y1=0.0, y2=1.0, steps=100):
    return lambda x: ((1 - math.cos(x * math.pi / steps)) / 2) * (y2 - y1) + y1

if params['cos_lr']:
    lf = one_cycle(1, params['lrf'], params['epochs'])  # cosine
else:
    lf = lambda x: (1 - x / params['epochs']) * (1.0 - params['lrf']) + params['lrf']  # linear

# stride를 모델에서 추출 (DataParallel로 감싸기 전)
stride = max(int(baseline_model.model.stride.max()), 32)

import yaml
# with open('/home/smile/work/pbs06/yolov5/data/hyps/hyp.scratch-custom-1117-RBCImage.yaml') as f:
#     hyp = yaml.safe_load(f)

with open('/home/smile/work/pbs06/yolov10/data/hyps/default.yaml') as f:
    hyp = yaml.safe_load(f)

# 모델을 DataParallel로 감싸기
baseline_model = torch.nn.DataParallel(baseline_model, device_ids=[0, 1])
baseline_model = baseline_model.cuda()

# 조기 중단 설정
stopper, stop = EarlyStopping(patience=params['patience']), False

import os
import datetime
import re  # 추가

# 현재 날짜를 'YYYYMMDD' 형식으로 가져오기
date_str = datetime.datetime.now().strftime('%y%m%d')

# 저장할 디렉토리 경로 설정
base_save_dir = '/home/smile/work/pbs/model/yolov5/runsbackup_v10'

# 해당 날짜의 기존 디렉토리 목록 확인
existing_dirs = [
    d for d in os.listdir(base_save_dir)
    if os.path.isdir(os.path.join(base_save_dir, d)) and re.match(rf"{date_str}_U2L_\d{{3}}_v10", d)
]

# 가장 큰 번호 추출
max_index = 0
for dirname in existing_dirs:
    match = re.search(rf"{date_str}_U2L_(\d{{3}})_v10", dirname)
    if match:
        index = int(match.group(1))
        max_index = max(max_index, index)

# 다음 번호 생성
next_index = max_index + 1
index_str = f"{next_index:03d}"
new_save_dir = f"{date_str}_U2L_{index_str}_v10"

# 전체 경로 생성
full_save_path = os.path.join(base_save_dir, new_save_dir)

# 디렉토리 생성 (이미 존재하면 예외 발생)
os.makedirs(full_save_path, exist_ok=False)

# 클래스 명 파일 경로
class_file_path = os.path.join(full_save_path, 'label.txt')

# 클래스 명을 콤마로 구분된 한 줄로 파일에 기록
with open(class_file_path, 'w') as file:
    file.write(','.join(data_yaml['names']))

print(f'클래스 명 파일이 {class_file_path}에 저장되었습니다.')

# 결과를 저장할 디렉토리 경로를 업데이트
params['project'] = full_save_path

# 최고 precision 값을 저장할 변수
best_precision = 0
best_epoch = 0

# 전체 학습을 진행하며 매 에폭마다 precision을 확인
results = baseline_model.module.train(
    data=params['data'],
    epochs=params['epochs'],  # 전체 에폭 설정
    imgsz=params['imgsz'],
    optimizer=params['optimizer'],
    lr0=params['lr0'],
    lrf=params['lrf'],
    cos_lr=params['cos_lr'],
    label_smoothing=params['label_smoothing'],
    patience=params['patience'],
    freeze=params['freeze'],
    save_period=100,  # 각 에폭마다 모델 저장
    seed=params['seed'],
    batch=params['batch'],
    project=params['project'],
    name="blood_cell"
)

# 각 에폭 결과에서 precision 값을 확인하여 최고값 추적
for epoch in range(params['epochs']):
    current_precision = results['metrics']['precision']

    if current_precision > best_precision:
        best_precision = current_precision
        best_epoch = epoch + 1  # 에폭은 1부터 시작하므로 +1

        # 최고 precision 모델을 저장할 경로 설정
        best_model_path = os.path.join(full_save_path, f'best_precision_epoch_{best_epoch}.pt')
        baseline_model.module.save(best_model_path)

        print(f"에폭 {best_epoch}에서 최고 precision {best_precision:.4f}을 기록하여 모델을 저장했습니다: {best_model_path}")

클래스 명 파일이 /home/smile/work/pbs/model/yolov5/runsbackup_v10/251020_U2L_002_v10/label.txt에 저장되었습니다.
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/smile/work/pbs06/yolov5/data/blood_cell.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=800, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=[0], half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=384, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0036365501465160546, lrf=0.0006434696163064856, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10s.pt, momentum=0.937, mosaic=1.0, multi_scale=F

KeyboardInterrupt: 

클래스 명 파일이 /home/smile/work/pbs/model/yolov5/runsbackup_v10/251020_U2L_001_v10/label.txt에 저장되었습니다.
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/smile/work/pbs06/yolov5/data/blood_cell.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=800, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=[0], half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=384, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0036365501465160546, lrf=0.0006434696163064856, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10s.pt, momentum=0.937, mosaic=1.0, multi_scale=F

In [ ]:
import yaml

# YAML 파일 생성
new_data_yaml = {
    'train': '/home/smile/work/pbs06/train/images',
    'val': '/home/smile/work/pbs06/val/images',
    'test': '/home/smile/work/pbs06/test/images',
    'nc': 25,
    'names': [ 'BandNeutrophil', 'Basophil', 'Blast', 'Echinocyte', 'Elliptocyte', 'Eosinophil', 'GiantPlt', 'Lymphocyte', 'Monocyte', 'Myelocyte', 'Nucleated', 'PLT', 'RBC', 'Reticulocyte', 'Schistocyte', 'SegNeutrophil', 'Smudge', 'Stomatocyte', 'TargetCell', 'TearDropCell', 'ToxicGranule', 'ToxicVacuole', 'clump-PLT', 'degeneWBC', 'hyperSeg.'] 

}

# YAML 파일 경로
yaml_file_path = '/home/smile/work/pbs06/yolov5/data/new_blood_cell.yaml'

# 파일 저장
with open(yaml_file_path, 'w') as f:
    yaml.dump(new_data_yaml, f)

print(f"YAML 파일이 {yaml_file_path}에 저장되었습니다.")

from ultralytics import YOLO

# 모델 로드
best_model = YOLO('/home/smile/work/pbs/model/yolov5/runsbackup_v10/250527_U2L_001_v10/blood_cell3/weights/best.pt')

# 모델 평가 수행
test_results = best_model.val(data=yaml_file_path, split='test')
# 모델 평가 수행
# test_results = best_model.val(data=yaml_file_path, split='train')

# 평가 결과 출력
print(test_results)

In [ ]:
import yaml

# YAML 파일 생성
new_data_yaml = {
    'train': '/home/smile/work/pbs06/train/images',
    'val': '/home/smile/work/pbs06/val/images',
    'test': '/home/smile/work/pbs06/test/images',
    'nc': 28,
    'names': [
        "PLT", "clump-PLT", "GiantPlt", "RBC", "Echinocyte", "Elliptocyte", 
        "TearDropCell", "TargetCell", "Schistocyte", "Nucleated", "Reticulocyte",
        "SegNeutrophil", "Lymphocyte", "Monocyte", "Eosinophil", "Basophil", 
        "BandNeutrophil", "Blast", "ToxicGranule", "Myelocyte", "ToxicVacuole", 
        "hyperSeg.", "degeneWBC", "Smudge", "Granule-Lymp.", "Reactive-Lymp.", 
        "Abnormal-Lymp.", "Stomatocyte"
    ]
}

# YAML 파일 경로
yaml_file_path = '/home/smile/work/pbs06/yolov5/data/new_blood_cell.yaml'

# 파일 저장
with open(yaml_file_path, 'w') as f:
    yaml.dump(new_data_yaml, f)

print(f"YAML 파일이 {yaml_file_path}에 저장되었습니다.")

from ultralytics import YOLO

# 모델 로드
best_model = YOLO('/home/smile/work/pbs/model/yolov5/runsbackup_v10/250110_U2L_001_v10/blood_cell/weights/best.pt')

# 모델 평가 수행
test_results = best_model.val(data=yaml_file_path, split='test',conf = 0.75)
# 모델 평가 수행
# test_results = best_model.val(data=yaml_file_path, split='train')

# 평가 결과 출력
print(test_results)

In [ ]:
import torch
from ultralytics import YOLO

# 학습된 모델의 최적 파라미터를 로드
best_model = YOLO('/home/smile/work/pbs/model/yolov5/runsbackup_v10/241107_U2L_001_v10/blood_cell/weights/best.pt')

# 기존 데이터셋 YAML 파일 사용 (테스트 데이터 경로가 이 YAML 파일에 정의되어 있어야 합니다)
data_yaml_path = params['data']  # 이전에 사용한 YAML 파일 경로

# 테스트 데이터셋에 대해 모델 평가 수행 (confidence threshold를 0.75로 설정)
test_results = best_model.val(data=data_yaml_path, split='test')

# 평가 결과 출력
print(test_results)